In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes")

print("Path to dataset files:", path)

Path to dataset files: /Users/ddevatha/.cache/kagglehub/datasets/khalidsiddiqui2003/dfu-dataset-annotated-into-4-classes/versions/1


In [2]:
import torchvision
import torchvision.transforms as T

transforms = T.Compose([
    T.Resize((64, 64)),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [3]:
import numpy as np

data = torchvision.datasets.ImageFolder(root='./archive/train', transform=transforms)

X = list()
y = list()

for images, labels in data:
    X.append(images)
    y.append(labels)

In [ ]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67, shuffle=True)

X_train = torch.stack(X_train).float()
y_train = torch.tensor(y_train, dtype=torch.int64)

X_test = torch.stack(X_test).float()
y_test = torch.tensor(y_test, dtype=torch.int64)

train_data = TensorDataset(X_train, y_train)
test_data = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class UlcerNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3) # (3, 64, 64) -> (32, 62, 62)
        self.pool = nn.MaxPool2d(2, 2) # (32, 62, 62) -> (32, 31, 31)
        self.conv2 = nn.Conv2d(32, 64, 3) # (32, 31, 31) -> (64, 29, 29)
        # pooled: (64, 29, 29) -> (64, 14, 14)

        self.fc1 = nn.Linear(64 * 14 * 14, 32)
        self.fc2 = nn.Linear(32, 16)
        self.out = nn.Linear(16, 4)

    def forward(self, x):
        out = self.pool(F.relu(self.conv1(x)))
        out = self.pool(F.relu(self.conv2(out)))

        out = out.flatten(1)
        out = F.relu(self.fc1(out))
        out = F.relu(self.fc2(out))

        out = self.out(out)
        return out   

In [ ]:
model = UlcerNN()
model.train()

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

for epoch in range(20):
    total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1}: Loss >>> {avg_loss:.4f}')

# Save model
torch.save(model.state_dict(), 'ulcer_model_state.pth')
wandb.save('ulcer_model_state.pth')

Epoch 1: Loss >>> 1.1836
Epoch 2: Loss >>> 1.0253
Epoch 3: Loss >>> 0.8969
Epoch 4: Loss >>> 0.7555
Epoch 5: Loss >>> 0.6190
Epoch 6: Loss >>> 0.4969
Epoch 7: Loss >>> 0.3888
Epoch 8: Loss >>> 0.2987
Epoch 9: Loss >>> 0.2388
Epoch 10: Loss >>> 0.1727
Epoch 11: Loss >>> 0.1335
Epoch 12: Loss >>> 0.1187
Epoch 13: Loss >>> 0.1126
Epoch 14: Loss >>> 0.0905
Epoch 15: Loss >>> 0.0875
Epoch 16: Loss >>> 0.0627
Epoch 17: Loss >>> 0.0768
Epoch 18: Loss >>> 0.0421
Epoch 19: Loss >>> 0.0343
Epoch 20: Loss >>> 0.0275
Model saved to ulcer_model_state.pth


In [ ]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)

        loss = loss_function(output, labels)

        _, predictions = torch.max(output, 1)

        all_preds.extend(predictions)
        all_labels.extend(labels)

print(classification_report(all_labels, all_preds, target_names=['Grade 1','Grade 2','Grade 3','Grade 4']))

              precision    recall  f1-score   support

     Grade 1       0.94      0.81      0.87       468
     Grade 2       0.86      0.91      0.88       439
     Grade 3       0.82      0.88      0.85       538
     Grade 4       0.89      0.88      0.88       483

    accuracy                           0.87      1928
   macro avg       0.88      0.87      0.87      1928
weighted avg       0.88      0.87      0.87      1928

